In [ ]:
# !pip install numpy==1.23.1
## numpy 변경 절대 금지!!
import os
from dotenv import load_dotenv

# .env 파일 로딩
load_dotenv()
# 환경변수 불러오기
hf_token = os.getenv("hf_token")
sentiment_dataset = os.getenv("sentiment_dataset")
output_dir = os.getenv("output_dir")
logging_dir = os.getenv("logging_dir")
model_dir = os.getenv("model_dir")

  Using cached numpy-1.23.1.tar.gz (10.7 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Operation cancelled by user
^C


KeyboardInterrupt: 

In [ ]:
# Log in to your W&B account
import wandb
import random
import math

In [ ]:
wandb.login()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: danggu-su (codi_dankook) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
wandb.init(project='fact_tory', name='kobert_finetuning')

In [ ]:
from huggingface_hub import login, upload_file

# Hugging Face Access Token 및 리포지토리 이름 설정
repo_name = "Dang-gu/oss_sentiment"

# Hugging Face 로그인
login(token=hf_token)

In [ ]:
import numpy as np ## 버전 오류로 인한 에러 해결
np.bool = np.bool_

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import gluonnlp as nlp
from torch.utils.data import DataLoader, Dataset

# from transformers import AutoTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from tqdm.notebook import tqdm

import random

In [ ]:
# GPU를 사용할 것이므로, CUDA 세팅
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
from kobert_tokenizer import KoBERTTokenizer
from transformers import BertModel, AutoModelForSequenceClassification

# KoBERT 토크나이저 & 모델
tokenizer = KoBERTTokenizer.from_pretrained('skt/kobert-base-v1')
bertmodel = BertModel.from_pretrained('skt/kobert-base-v1', return_dict=False)

model = AutoModelForSequenceClassification.from_pretrained(
    "skt/kobert-base-v1",
    num_labels=3     # 🔁 클래스 수 꼭 맞춰줘 (negative/neutral/positive)
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLNetTokenizer'. 
The class this function is called from is 'KoBERTTokenizer'.


config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at skt/kobert-base-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import pandas as pd

data = pd.read_csv(sentiment_dataset)
data.head(5)

,labels,sentence,kor_sentence
0,neutral,"According to Gran, the company has no plans to...","Gran에 따르면, 그 회사는 회사가 성장하고 있는 곳이지만, 모든 생산을 러시아로..."
1,neutral,Technopolis plans to develop in stages an area...,테크노폴리스는 컴퓨터 기술과 통신 분야에서 일하는 회사들을 유치하기 위해 10만 평...
2,negative,The international electronic industry company ...,"국제 전자산업 회사인 엘코텍은 탈린 공장에서 수십 명의 직원을 해고했으며, 이전의 ..."
3,positive,With the new production plant the company woul...,새로운 생산공장으로 인해 회사는 예상되는 수요 증가를 충족시킬 수 있는 능력을 증가...
4,positive,According to the company's updated strategy fo...,"2009-2012년 회사의 업데이트된 전략에 따르면, Basware는 20% - 4..."


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

# 1. 라벨 숫자화
label_map = {"negative": 0, "neutral": 1, "positive": 2}
data["label"] = data["labels"].map(label_map)

# 2. Dataset 객체로 변환 + train/test 분리
dataset = Dataset.from_dict(data).train_test_split(test_size=0.2)

# 3. tokenizer 준비
tokenizer = AutoTokenizer.from_pretrained("skt/kobert-base-v1")

# 4. 토크나이저 함수 정의
def tokenize_function(example):
    return tokenizer(
        example["kor_sentence"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

# 5. 데이터셋에 tokenizer 적용
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/3876 [00:00<?, ? examples/s]

Map:   0%|          | 0/970 [00:00<?, ? examples/s]

In [ ]:
# 6. torch tensor로 변환
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
tokenized_dataset['train']

Dataset({
    features: ['labels', 'sentence', 'kor_sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3876
})

In [ ]:
#evalution compute_metrics 선언

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=output_dir,          # output directory
    report_to='wandb',                          # wandb 사용
    num_train_epochs=15,                        # 학습 epoch 설정
    per_device_train_batch_size=32,             # train batch_size 설정
    per_device_eval_batch_size=32,
    logging_dir=logging_dir,            # directory for storing logs
    logging_steps=100,                          # 학습 log 기록 단위
    save_total_limit=2,                         # 학습 결과 저장 최대 개수
    eval_strategy="epoch",                # 매 epoch마다 평가 실행
    save_strategy="epoch",                      # 매 epoch마다 모델 저장
    load_best_model_at_end=True,                # 가장 성능이 좋은 모델을 마지막에 load
)

# 최적화 알고리즘(optimizer) 설정
optimizer = AdamW(model.parameters(), lr=2e-5)


#initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    optimizers=(optimizer, None),
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

# 모델 학습
trainer.train()

## 과적합 확인 -> 더 필요함

/tmp/ipython-input-22-2662795146.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.834100,0.813235,0.663918,0.589645,0.556546,0.663918
2,0.832500,0.813260,0.654639,0.593297,0.551829,0.654639
3,0.785400,0.833022,0.650515,0.598252,0.556233,0.650515
4,0.764000,0.780205,0.668041,0.597417,0.562214,0.668041
5,0.709200,0.805983,0.655670,0.615779,0.618755,0.655670
6,0.690200,0.848152,0.661856,0.628684,0.625780,0.661856
7,0.664100,0.842049,0.659794,0.614148,0.619881,0.659794
8,0.625200,0.893132,0.652577,0.627382,0.622270,0.652577
9,0.596600,0.902696,0.628866,0.624562,0.620933,0.628866
10,0.518900,0.941764,0.658763,0.637764,0.632075,0.658763


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


TrainOutput(global_step=1830, training_loss=0.5933582748871683, metrics={'train_runtime': 1482.3558, 'train_samples_per_second': 39.221, 'train_steps_per_second': 1.235, 'total_flos': 3824353526676480.0, 'train_loss': 0.5933582748871683, 'epoch': 15.0})

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

model_checkpoint = "/checkpoint"

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=3     # 🔁 클래스 수 꼭 맞춰줘 (negative/neutral/positive)
)

In [ ]:
# 테스트(Test) 데이터셋 평가
trainer.evaluate(tokenized_dataset['test'])

{'eval_loss': 0.7802051305770874,
 'eval_accuracy': 0.668041237113402,
 'eval_f1': 0.5974166244530936,
 'eval_precision': 0.5622140014030379,
 'eval_recall': 0.668041237113402,
 'eval_runtime': 6.0373,
 'eval_samples_per_second': 160.668,
 'eval_steps_per_second': 5.135,
 'epoch': 15.0}

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

username = "Dang-gu"
MODEL_NAME = 'kobert_sentiment_finetuning'

api.create_repo(
    token=hf_token,
    repo_id=f"{username}/{MODEL_NAME}",
    repo_type="model"
)

In [ ]:
from huggingface_hub import HfApi, HfFolder, upload_folder

# 사용자 정보
username = "Dang-gu"  # 🤚 너의 HF 계정명 (주의: 실제로는 소문자/하이픈 허용)
model_name = "kobert_sentiment_finetuning"
# 전체 경로
repo_id = 'Dang-gu/oss_sentiment'

# HF API 객체 생성
api = HfApi()

# 토큰은 notebook_login으로 이미 저장되었으므로 생략 가능
# 아니면 아래처럼 직접 삽입 가능 (권장 X)
# HfFolder.save_token("hf_XXX")

# 🔧 모델 업로드
upload_folder(
    folder_path=model_dir,
    repo_id=repo_id,
    repo_type="model",
    commit_message="upload trained KoBERT sentiment classifier",
    ignore_patterns=[".ipynb_checkpoints"]
)

Upload 6 LFS files:   0%|          | 0/6 [00:00<?, ?it/s]

rng_state.pth:   0%|          | 0.00/14.2k [00:00<?, ?B/s]

scheduler.pt:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

optimizer.pt:   0%|          | 0.00/738M [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.37k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Dang-gu/oss_sentiment/commit/ee864808f17194567ccf4f28f0c0e9eae4bc8df5', commit_message='upload trained KoBERT sentiment classifier', commit_description='', oid='ee864808f17194567ccf4f28f0c0e9eae4bc8df5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Dang-gu/oss_sentiment', endpoint='https://huggingface.co', repo_type='model', repo_id='Dang-gu/oss_sentiment'), pr_revision=None, pr_num=None)

In [ ]:
## 실제 기사로 판단